# Tab Transformer w/ Pseudo-Labels, Claude Vibe-coding

Based on my XGB notebook, which in turn is based on Chris Deotte's excellent notebook here: https://www.kaggle.com/code/cdeotte/first-place-single-model-lb-38-81

I did this last playground competition too: 

https://www.kaggle.com/code/include4eto/tabtransfomer-chatgpt-vibe-coding

This time I used Claude Opus 4.5 instead of ChatGPT to vibe code it:

- I gave it my XGB notebook from this playground series +
- my notebook from the old playground series
- and asked it to convert the XGB one to use a TabTransformer

In [1]:
NAME = '002'

In [2]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
import tensorflow as tf
import keras
import seaborn as sns

2026-04-18 16:27:27.813702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776529648.047805      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776529648.111016      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776529648.633993      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776529648.634033      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776529648.634035      24 computation_placer.cc:177] computation placer alr

In [3]:
%load_ext cudf.pandas

import numpy as np, pandas as pd, gc
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 200)

In [4]:
train_file = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
test_file = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
original_file = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
orig = pd.read_csv(original_file)

train_ids = train['id'].copy()
test_ids = test['id'].copy() # for submission

In [5]:
train['Irrigation_Need'] = train['Irrigation_Need'].map(_classes := {'Low': 0, 'Medium': 1, 'High': 2})
orig['Irrigation_Need'] = orig['Irrigation_Need'].map(_classes)

In [6]:
TARGET = 'Irrigation_Need'
NUMS = list(train.drop(['id', TARGET], axis=1)._get_numeric_data().columns)
CATS = [c for c in train.drop(['id', TARGET], axis=1).columns if c not in NUMS]

In [7]:
# simple check
# id + target = 2
assert len(NUMS) + len(CATS) + 2 == train.shape[1]

In [8]:
print(f"There are {len(CATS)} categorical columns:")
print(CATS)
print(f"There are {len(NUMS)} numerical columns:")
print(NUMS)

There are 8 categorical columns:
['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
There are 11 numerical columns:
['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


# Feature Engineering

In [9]:
NEW_NUMS = []
NEW_CATS = []
NUM_AS_CAT = []
TO_REMOVE = []
NON_TE_CATS = []

In [10]:
_to_combo = CATS

for i, c1 in enumerate(_to_combo[:-1]):
    for j, c2 in enumerate(_to_combo[i + 1:]):
        _new_col = f'COMBO_{c1}_{c2}'

        for df in [train, test, orig]:
            df[_new_col] = df[c1].astype('str') + '_' + df[c2].astype('str')
        NEW_CATS.append(_new_col)

        # 3-combos
        # for k, c3 in enumerate(_to_combo[i+j+2:]):
        #     _new_col = f'COMBO_{c1}_{c2}_{c3}'
        #     for df in [train, test, orig]:
        #         df[_new_col] = df[c1].astype('str') + '_' + df[c2].astype('str') + '_' + df[c3].astype('str')
    
        #     NEW_CATS.append(_new_col)

            # print(c1, c2, c3)
            # 4-combos
            # for l, c4 in enumerate(_to_combo[i+j+k+3:]):
            #     _new_col = f'COMBO_{c1}_{c2}_{c3}_{c4}'
            #     for df in [train, test, orig]:
            #         df[_new_col] = df[c1].astype('str') + '_' + df[c2].astype('str') + '_' + df[c3].astype('str') + '_' + df[c4].astype('str')
        
            #     NEW_CATS.append(_new_col)

In [11]:
# Compute frequencies from ALL data (train + orig + test)
for cat in CATS + NEW_CATS:
    freq = pd.concat([train[cat], orig[cat], test[cat]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{cat}'] = df[cat].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{cat}')

In [12]:
# numerical as categorical
for col in NUMS:    
    _new_col = f'CAT_{col}'
    NUM_AS_CAT.append(_new_col)

    for df in [train, test, orig]:
        df[_new_col] = df[col].astype(str).astype('category')

In [13]:
# adapted from https://www.kaggle.com/code/yunsuxiaozi/pss6e4-xgb-cv-0-979805

M = train[NUMS].max()
DIGIT_FEATURES = []

for c in NUMS:
    for df in [train, test]:
        for k in range(-4,4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')
            DIGIT_FEATURES += [f"{c}_digit{k}"]
    
        if M[c]<10:
            df[c]=df[c].round(3)
        elif M[c]<100:
            df[c]=df[c].round(2)
        else:
            df[c]=df[c].round(1)


DROP=[c for c in test.columns if test[c].nunique()==1]
print(f"DROP:{DROP}")

train.drop(DROP,axis=1,inplace=True)
test.drop(DROP,axis=1,inplace=True)

DIGIT_FEATURES = list(set(DIGIT_FEATURES) - set(DROP))
print()
print('=' * 50)
print("DIGITS: ", DIGIT_FEATURES)
NEW_CATS += DIGIT_FEATURES

DROP:['Soil_pH_digit1', 'Soil_pH_digit2', 'Soil_pH_digit3', 'Soil_Moisture_digit2', 'Soil_Moisture_digit3', 'Organic_Carbon_digit1', 'Organic_Carbon_digit2', 'Organic_Carbon_digit3', 'Electrical_Conductivity_digit1', 'Electrical_Conductivity_digit2', 'Electrical_Conductivity_digit3', 'Temperature_C_digit2', 'Temperature_C_digit3', 'Humidity_digit2', 'Humidity_digit3', 'Sunlight_Hours_digit2', 'Sunlight_Hours_digit3', 'Wind_Speed_kmh_digit2', 'Wind_Speed_kmh_digit3', 'Field_Area_hectare_digit2', 'Field_Area_hectare_digit3', 'Previous_Irrigation_mm_digit3']

DIGITS:  ['Field_Area_hectare_digit-4', 'Wind_Speed_kmh_digit-1', 'Organic_Carbon_digit-2', 'Previous_Irrigation_mm_digit-3', 'Soil_pH_digit-2', 'Wind_Speed_kmh_digit0', 'Rainfall_mm_digit1', 'Soil_pH_digit-3', 'Temperature_C_digit-3', 'Electrical_Conductivity_digit-3', 'Rainfall_mm_digit0', 'Humidity_digit-4', 'Temperature_C_digit-4', 'Sunlight_Hours_digit-4', 'Soil_pH_digit0', 'Soil_Moisture_digit1', 'Temperature_C_digit1', 'Previo

see https://www.kaggle.com/code/cdeotte/original-data-exact-formula for exact formula for `orig`

In [14]:
for df in [train, test]:
    # 4 boolean numeric features based on threshold insights
    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype(int)
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype(int)
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype(int)


TRES_CATS = ['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10']
NEW_CATS += TRES_CATS

In [15]:
for df_ in [train, test]:
    df = pd.get_dummies(
        df_[NUMS + CATS + TRES_CATS],
        columns=CATS,
        drop_first=False
    )

    df_['logit(P(y=Low))'] = 16.3173 + (-11.0237 * df["soil_lt_25"]) + (-5.8559 * df["temp_gt_30"]) + (-10.8500 * df["rain_lt_300"]) + (-5.8284 * df["wind_gt_10"]) + (-5.4155 * df["Crop_Growth_Stage_Flowering"]) + (5.5073 * df["Crop_Growth_Stage_Harvest"]) + (5.2299 * df["Crop_Growth_Stage_Sowing"]) + (-5.4617 * df["Crop_Growth_Stage_Vegetative"]) + (-3.0014 * df["Mulching_Used_No"]) + (2.8613 * df["Mulching_Used_Yes"])
    df_['logit(P(y=Medium))'] = 4.6524 + (0.3290 * df["soil_lt_25"]) + (-0.0204 * df["temp_gt_30"]) + (0.1542 * df["rain_lt_300"]) + (0.0841 * df["wind_gt_10"]) + (0.3586 * df["Crop_Growth_Stage_Flowering"]) + (-0.1348 * df["Crop_Growth_Stage_Harvest"]) + (-0.3547 * df["Crop_Growth_Stage_Sowing"]) + (0.3334 * df["Crop_Growth_Stage_Vegetative"]) + (0.1883 * df["Mulching_Used_No"]) + (0.0142 * df["Mulching_Used_Yes"])
    df_['logit(P(y=High))'] = -20.9697 + (10.6947 * df["soil_lt_25"]) + (5.8763 * df["temp_gt_30"]) + (10.6958 * df["rain_lt_300"]) + (5.7444 * df["wind_gt_10"]) + (5.0569 * df["Crop_Growth_Stage_Flowering"]) + (-5.3725 * df["Crop_Growth_Stage_Harvest"]) + (-4.8752 * df["Crop_Growth_Stage_Sowing"]) + (5.1283 * df["Crop_Growth_Stage_Vegetative"]) + (2.8131 * df["Mulching_Used_No"]) + (-2.8755 * df["Mulching_Used_Yes"])

NEW_NUMS += ['logit(P(y=Low))', 'logit(P(y=Medium))', 'logit(P(y=High))']

In [16]:
train.isna().any().any(), test.isna().any().any()

(np.False_, np.False_)

In [17]:
for col in CATS + NUMS:
    stats = orig.groupby(col)[TARGET].agg(['mean', 'std']).reset_index()
    stats.columns = [col] + [f"ORIG_{col}_{s}" for s in ['mean', 'std']]

    train = train.merge(stats, on=col, how='left')
    test = test.merge(stats, on=col, how='left')

    fill_values = {
        f"ORIG_{col}_mean": 0.5,
        f"ORIG_{col}_std": 0,
    }
    train = train.fillna(value=fill_values)
    test = test.fillna(value=fill_values)

    NEW_NUMS.extend([f"ORIG_{col}_{s}" for s in ['mean', 'std']])

In [18]:
FEATURES = NUMS + CATS + NEW_NUMS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS
print(f'We now have {len(FEATURES)} columns:')
print(FEATURES)

We now have 205 columns:
['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm', 'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'FREQ_Soil_Type', 'FREQ_Crop_Type', 'FREQ_Crop_Growth_Stage', 'FREQ_Season', 'FREQ_Irrigation_Type', 'FREQ_Water_Source', 'FREQ_Mulching_Used', 'FREQ_Region', 'FREQ_COMBO_Soil_Type_Crop_Type', 'FREQ_COMBO_Soil_Type_Crop_Growth_Stage', 'FREQ_COMBO_Soil_Type_Season', 'FREQ_COMBO_Soil_Type_Irrigation_Type', 'FREQ_COMBO_Soil_Type_Water_Source', 'FREQ_COMBO_Soil_Type_Mulching_Used', 'FREQ_COMBO_Soil_Type_Region', 'FREQ_COMBO_Crop_Type_Crop_Growth_Stage', 'FREQ_COMBO_Crop_Type_Season', 'FREQ_COMBO_Crop_Type_Irrigation_Type', 'FREQ_COMBO_Crop_Type_Water_Source', 'FREQ_COMBO_Crop_Type_Mulching_Used', 'FREQ_COMBO_Crop_Type_Region', 'FREQ_COMBO_Crop_Growth

In [19]:
TE_COLUMNS = NUM_AS_CAT + CATS + NEW_CATS
TO_REMOVE += NUM_AS_CAT + CATS + NEW_CATS
QUANTILE_COLUMNS = []

# Model Training

In [20]:
np.random.seed(11)

In [21]:
PSEUDO_LABELS = False
TRES = 0.999

FOLDS = 5
INNER_FOLDS = 5

In [22]:
# TabTransformer hyperparameters
EMBED_DIM = 16
NUM_HEADS = 4
FF_DIM = 64
NUM_TRANSFORMER_BLOCKS = 2
MLP_HIDDEN = (128, 64)
DROPOUT = 0.2
LEARNING_RATE = 1e-3
BATCH_SIZE = 4096
EPOCHS = 100
SEED = 11

keras.utils.set_random_seed(SEED)

In [23]:
# ============================================================
# CATEGORICAL VOCAB (fitted on train+test combined)
# ============================================================
ALL_CATS = CATS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS
ALL_NUMS_MODEL = [c for c in NUMS + NEW_NUMS if c not in ALL_CATS]

cat_vocab = {}
for c in ALL_CATS:
    vals = pd.concat([train[c], test[c]], axis=0).astype(str).unique().tolist()
    cat_vocab[c] = sorted(vals)

print(f"TabTransformer categorical cols: {len(ALL_CATS)}")
print(f"TabTransformer numerical cols:   {len(ALL_NUMS_MODEL)}")


# ============================================================
# BUILD TABTRANSFORMER (multiclass, 3 classes)
# ============================================================
def build_tabtransformer(
    cat_cols,
    num_col_names,
    cat_vocab,
    te_dim,
    n_classes=3,
    embed_dim=16,
    num_heads=4,
    ff_dim=64,
    num_transformer_blocks=2,
    mlp_hidden=(128, 64),
    dropout=0.2,
    lr=1e-3,
):
    inputs = {}
    cat_embeds = []

    # categorical inputs -> StringLookup -> Embedding
    for c in cat_cols:
        inp = keras.Input(shape=(1,), name=c, dtype="string")
        inputs[c] = inp

        lookup = keras.layers.StringLookup(
            vocabulary=cat_vocab[c],
            mask_token=None,
            num_oov_indices=1,
            output_mode="int",
        )
        x = lookup(inp)
        x = keras.layers.Embedding(
            input_dim=lookup.vocabulary_size(),
            output_dim=embed_dim,
        )(x)
        x = keras.layers.Reshape((embed_dim,))(x)
        cat_embeds.append(x)

    # Transformer over categorical embeddings
    if cat_embeds:
        cat_tokens = keras.layers.Lambda(lambda xs: tf.stack(xs, axis=1))(cat_embeds)

        x_cat = cat_tokens
        for _ in range(num_transformer_blocks):
            attn_out = keras.layers.MultiHeadAttention(
                num_heads=num_heads,
                key_dim=embed_dim,
                dropout=dropout,
            )(x_cat, x_cat)
            x_cat = keras.layers.Add()([x_cat, attn_out])
            x_cat = keras.layers.LayerNormalization(epsilon=1e-6)(x_cat)

            ff = keras.layers.Dense(ff_dim, activation="gelu")(x_cat)
            ff = keras.layers.Dropout(dropout)(ff)
            ff = keras.layers.Dense(embed_dim)(ff)
            x_cat = keras.layers.Add()([x_cat, ff])
            x_cat = keras.layers.LayerNormalization(epsilon=1e-6)(x_cat)

        x_cat = keras.layers.Flatten()(x_cat)

    # numeric branch
    num_features = []
    for c in num_col_names:
        inp = keras.Input(shape=(1,), name=c, dtype="float32")
        inputs[c] = inp
        num_features.append(inp)

    # TE branch
    te_inp = keras.Input(shape=(te_dim,), name="te_feats", dtype="float32")
    inputs["te_feats"] = te_inp
    te_x = keras.layers.BatchNormalization()(te_inp)
    te_x = keras.layers.Dense(min(256, max(32, te_dim // 2)))(te_x)
    te_x = keras.layers.Activation("relu")(te_x)
    te_x = keras.layers.Dropout(dropout)(te_x)

    feats = []
    if cat_embeds:
        feats.append(x_cat)
    feats.append(te_x)
    if num_features:
        nums = keras.layers.Concatenate()(num_features) if len(num_features) > 1 else num_features[0]
        feats.append(nums)

    x = keras.layers.Concatenate()(feats) if len(feats) > 1 else feats[0]

    # MLP head
    for h in mlp_hidden:
        x = keras.layers.Dense(h)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
        x = keras.layers.Dropout(dropout)(x)

    out = keras.layers.Dense(n_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    return model


def make_model_inputs(df, cat_cols, num_col_names, te_features):
    d = {}
    for c in cat_cols:
        d[c] = df[c].astype(str).values
    for c in num_col_names:
        d[c] = df[c].astype("float32").values.reshape(-1, 1)
    d["te_feats"] = te_features.astype(np.float32)
    return d

TabTransformer categorical cols: 117
TabTransformer numerical cols:   88


# TE

In [24]:
from functools import reduce

class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=(), target_col='target'):
        self.category_cols = category_cols
        self.classes_ = sorted(train[target_col].unique())
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        self.stats_ = {}

        for c in category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y = (train[target_col] == cls).astype(int)
                grp = train[[c]].assign(y=y.values)

                cum_cnt = grp.groupby(c, observed=False)['y'].cumcount()
                cum_sum = grp.groupby(c, observed=False)['y'].cumsum() - grp['y']

                prior = self.global_prior_[k]
                te = (cum_sum + self.a * prior) / (cum_cnt + self.a)
                te_col = f'{c}_TE_cls{cls}'
                train[te_col] = te.values

                agg = grp.groupby(c, observed=False)['y'].agg(count='count', total='sum').reset_index()
                agg.columns = [c, f'{c}_n_{cls}', f'{c}_s_{cls}']
                stats_list.append(agg)

            self.stats_[c] = reduce(lambda l, r: l.merge(r, on=c, how='outer'), stats_list)

        return train

    def transform(self, test):
        for c in self.category_cols:
            test = test.merge(self.stats_[c], on=c, how='left')

            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                n_col, s_col = f'{c}_n_{cls}', f'{c}_s_{cls}'
                prior = self.global_prior_[k]

                if n_col in test.columns:
                    test[te_col] = ((test[s_col] + self.a * prior) / (test[n_col] + self.a)).fillna(prior)
                    test.drop(columns=[n_col, s_col], inplace=True)
                else:
                    test[te_col] = prior

        return test

# Model Training

In [25]:
def metric(y_true, y_pred, sample_weight=None):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred, sample_weight=sample_weight)

In [26]:
print(f"\n{'='*80}")
print("TRAINING TABTRANSFORMER")
print("="*80)

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=11)

oof = np.zeros((len(train), 3))
pred = np.zeros((len(test), 3))
metric_folds = []
pred_per_fold = []

for i, (train_index, val_index) in enumerate(kf.split(train)):
    print(f"\nFold {i+1}/{FOLDS}")
    keras.utils.set_random_seed(SEED + i)

    # ===================================
    # TRAIN/VAL SPLIT
    # ===================================
    X_train = train.loc[train_index, FEATURES + [TARGET]].reset_index(drop=True).copy()
    y_train = train.loc[train_index, TARGET].values

    X_val = train.loc[val_index, FEATURES].reset_index(drop=True).copy()
    y_val = train.loc[val_index, TARGET].values

    X_test = test[FEATURES].reset_index(drop=True).copy()

    # ===================================
    # Target Encoding (OrderedTE)
    # ===================================
    target_encoder = OrderedTE(a=1)
    X_train = target_encoder.fit(X_train, category_cols=TE_COLUMNS, target_col=TARGET)
    X_val = target_encoder.transform(X_val)
    X_test = target_encoder.transform(X_test)

    # Drop raw categoricals and target
    X_train.drop(TO_REMOVE, axis=1, inplace=True)
    X_val.drop(TO_REMOVE, axis=1, inplace=True)
    X_test.drop(TO_REMOVE, axis=1, inplace=True)
    X_train = X_train.drop([TARGET], axis=1)
    COLS = X_train.columns.tolist()

    # Identify TE-generated columns (everything that's not in ALL_NUMS_MODEL and not in ALL_CATS)
    te_feat_cols = [c for c in COLS if c not in ALL_NUMS_MODEL and c not in ALL_CATS]
    model_num_cols = [c for c in ALL_NUMS_MODEL if c in COLS]
    model_cat_cols = [c for c in ALL_CATS if c in COLS]

    # Extract TE features as dense array
    tr_te = X_train[te_feat_cols].values.astype(np.float32)
    va_te = X_val[te_feat_cols].values.astype(np.float32)
    te_te = X_test[te_feat_cols].values.astype(np.float32)

    # Scale TE features
    scaler_te = StandardScaler()
    tr_te = scaler_te.fit_transform(tr_te).astype(np.float32)
    va_te = scaler_te.transform(va_te).astype(np.float32)
    te_te = scaler_te.transform(te_te).astype(np.float32)

    # Scale numeric features
    if model_num_cols:
        scaler_num = StandardScaler()
        X_train[model_num_cols] = scaler_num.fit_transform(X_train[model_num_cols])
        X_val[model_num_cols] = scaler_num.transform(X_val[model_num_cols])
        X_test[model_num_cols] = scaler_num.transform(X_test[model_num_cols])

    # ===================================
    # Build & train TabTransformer
    # ===================================
    te_dim = tr_te.shape[1]
    print(f"  cat cols: {len(model_cat_cols)}, num cols: {len(model_num_cols)}, TE dim: {te_dim}")

    model = build_tabtransformer(
        cat_cols=model_cat_cols,
        num_col_names=model_num_cols,
        cat_vocab=cat_vocab,
        te_dim=te_dim,
        n_classes=3,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        ff_dim=FF_DIM,
        num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
        mlp_hidden=MLP_HIDDEN,
        dropout=DROPOUT,
        lr=LEARNING_RATE,
    )

    # Compute sample weights for class imbalance
    s_wei_train = compute_sample_weight('balanced', y_train)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_acc",
            mode="max",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_acc",
            mode="max",
            factor=0.5,
            patience=5,
            min_lr=1e-5,
            verbose=1,
        ),
    ]

    model.fit(
        make_model_inputs(X_train, model_cat_cols, model_num_cols, tr_te),
        y_train,
        sample_weight=s_wei_train,
        validation_data=(
            make_model_inputs(X_val, model_cat_cols, model_num_cols, va_te),
            y_val,
        ),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=callbacks,
        verbose=2,
    )

    # ===================================
    # Predict OOF and TEST
    # ===================================
    oof[val_index] = model.predict(
        make_model_inputs(X_val, model_cat_cols, model_num_cols, va_te),
        batch_size=BATCH_SIZE, verbose=0,
    )
    bal_acc_fold = metric(y_val, oof[val_index])
    metric_folds.append(bal_acc_fold)
    print(f"Validation Balanced Accuracy: {bal_acc_fold:.5f}")

    _test_preds = model.predict(
        make_model_inputs(X_test, model_cat_cols, model_num_cols, te_te),
        batch_size=BATCH_SIZE, verbose=0,
    )
    pred += _test_preds
    pred_per_fold.append(_test_preds)

    # CLEAR MEMORY
    del X_train, X_val, X_test
    del y_train, y_val
    if i != FOLDS - 1:
        del model
    gc.collect()

pred /= FOLDS


TRAINING TABTRANSFORMER

Fold 1/5
  cat cols: 0, num cols: 88, TE dim: 351


I0000 00:00:1776529826.627687      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776529826.632921      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13753 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/100


I0000 00:00:1776529835.050682      78 service.cc:152] XLA service 0x7c96e400b8d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776529835.050729      78 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776529835.050734      78 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776529835.539753      78 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776529839.436732      78 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


124/124 - 14s - 112ms/step - acc: 0.8609 - loss: 0.3092 - val_acc: 0.8950 - val_loss: 0.4016 - learning_rate: 1.0000e-03
Epoch 2/100
124/124 - 2s - 15ms/step - acc: 0.9696 - loss: 0.1087 - val_acc: 0.9679 - val_loss: 0.1037 - learning_rate: 1.0000e-03
Epoch 3/100
124/124 - 2s - 16ms/step - acc: 0.9749 - loss: 0.0927 - val_acc: 0.9799 - val_loss: 0.0722 - learning_rate: 1.0000e-03
Epoch 4/100
124/124 - 2s - 16ms/step - acc: 0.9772 - loss: 0.0849 - val_acc: 0.9823 - val_loss: 0.0640 - learning_rate: 1.0000e-03
Epoch 5/100
124/124 - 2s - 16ms/step - acc: 0.9781 - loss: 0.0816 - val_acc: 0.9826 - val_loss: 0.0627 - learning_rate: 1.0000e-03
Epoch 6/100
124/124 - 2s - 16ms/step - acc: 0.9786 - loss: 0.0786 - val_acc: 0.9834 - val_loss: 0.0602 - learning_rate: 1.0000e-03
Epoch 7/100
124/124 - 2s - 16ms/step - acc: 0.9793 - loss: 0.0757 - val_acc: 0.9841 - val_loss: 0.0575 - learning_rate: 1.0000e-03
Epoch 8/100
124/124 - 2s - 15ms/step - acc: 0.9797 - loss: 0.0735 - val_acc: 0.9841 - val_los

In [27]:
print(f'Fold Balanced Accuracy {np.mean(metric_folds):.5f} +- {np.std(metric_folds):.5f}')

true = train[TARGET].values
print(f'Overall Balanced Accuracy: {metric(true, oof)}')

Fold Balanced Accuracy 0.97564 +- 0.00112
Overall Balanced Accuracy: 0.9756402045213107


In [28]:
print(f'\nIn total, we used {len(COLS)} features, Wow!\n')
print(list(COLS))


In total, we used 439 features, Wow!

['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm', 'FREQ_Soil_Type', 'FREQ_Crop_Type', 'FREQ_Crop_Growth_Stage', 'FREQ_Season', 'FREQ_Irrigation_Type', 'FREQ_Water_Source', 'FREQ_Mulching_Used', 'FREQ_Region', 'FREQ_COMBO_Soil_Type_Crop_Type', 'FREQ_COMBO_Soil_Type_Crop_Growth_Stage', 'FREQ_COMBO_Soil_Type_Season', 'FREQ_COMBO_Soil_Type_Irrigation_Type', 'FREQ_COMBO_Soil_Type_Water_Source', 'FREQ_COMBO_Soil_Type_Mulching_Used', 'FREQ_COMBO_Soil_Type_Region', 'FREQ_COMBO_Crop_Type_Crop_Growth_Stage', 'FREQ_COMBO_Crop_Type_Season', 'FREQ_COMBO_Crop_Type_Irrigation_Type', 'FREQ_COMBO_Crop_Type_Water_Source', 'FREQ_COMBO_Crop_Type_Mulching_Used', 'FREQ_COMBO_Crop_Type_Region', 'FREQ_COMBO_Crop_Growth_Stage_Season', 'FREQ_COMBO_Crop_Growth_Stage_Irrigation_Type', 'FREQ_COMBO_Crop_Growth_Stage_Water_Sourc

In [29]:
def accuracy_score(t,p):
    if len(p.shape)==2:
        p=np.argmax(p,axis=1)
    C=3
    acc=0.0
    for i in range(C):
        acc+=np.sum((t==i)&(p==i))/np.sum(t==i)/C
    return acc      

In [30]:
# see https://www.kaggle.com/code/yunsuxiaozi/pss6e4-xgb-cv-0-979805
oof_preds = oof
y = train[TARGET]

import optuna
from optuna.samplers import TPESampler

def objective(trial):
    cw1 = trial.suggest_float('cw1', 0.5, 3.0)
    cw2 = trial.suggest_float('cw2', 0.5, 3.0)
    cw3 = trial.suggest_float('cw3', 0.5, 3.0)
    
    class_weights = np.array([cw1, cw2, cw3])
    adjusted_probs = oof_preds * class_weights
    
    adjusted_probs = adjusted_probs / adjusted_probs.sum(axis=1, keepdims=True)
    acc = accuracy_score(y, np.argmax(adjusted_probs, axis=1))
    
    return acc

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='class_weight_optimization'
)

[I 2026-04-18 16:47:17,954] A new study created in memory with name: class_weight_optimization


In [31]:
study.optimize(objective, n_trials=200, show_progress_bar=True)

print(f"Best oof acc: {study.best_value:.6f}")
print(f"  class_0 = {study.best_params['cw1']:.4f}")
print(f"  class_1 = {study.best_params['cw2']:.4f}")
print(f"  class_2 = {study.best_params['cw3']:.4f}")

best_cw = np.array([study.best_params['cw1'], 
                    study.best_params['cw2'], 
                    study.best_params['cw3']])

final_oof_probs = oof * best_cw
final_oof_probs = final_oof_probs / final_oof_probs.sum(axis=1, keepdims=True)

final_test_probs = pred * best_cw
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)

final_test_preds = np.argmax(final_test_probs, axis=1)

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-04-18 16:47:18,090] Trial 0 finished with value: 0.975025608381969 and parameters: {'cw1': 1.4363502971184063, 'cw2': 2.87678576602479, 'cw3': 2.3299848545285125}. Best is trial 0 with value: 0.975025608381969.
[I 2026-04-18 16:47:18,151] Trial 1 finished with value: 0.9754988001043898 and parameters: {'cw1': 1.9966462104925915, 'cw2': 0.8900466011060912, 'cw3': 0.8899863008405067}. Best is trial 1 with value: 0.9754988001043898.
[I 2026-04-18 16:47:18,209] Trial 2 finished with value: 0.9736136254948697 and parameters: {'cw1': 0.6452090304204987, 'cw2': 2.665440364437338, 'cw3': 2.002787529358022}. Best is trial 1 with value: 0.9754988001043898.
[I 2026-04-18 16:47:18,262] Trial 3 finished with value: 0.9757735328064168 and parameters: {'cw1': 2.2701814444901136, 'cw2': 0.5514612357395061, 'cw3': 2.9247746304049858}. Best is trial 3 with value: 0.9757735328064168.
[I 2026-04-18 16:47:18,318] Trial 4 finished with value: 0.9752869879815155 and parameters: {'cw1': 2.581106602001

In [32]:
print(f"Best oof acc: {study.best_value:.6f}")
print(f"  class_0 = {study.best_params['cw1']:.4f}")
print(f"  class_1 = {study.best_params['cw2']:.4f}")
print(f"  class_2 = {study.best_params['cw3']:.4f}")

Best oof acc: 0.976366
  class_0 = 0.8689
  class_1 = 0.7890
  class_2 = 2.6500


In [33]:
print(f'Overall Balanced Accuracy: {metric(true, final_oof_probs)}')

Overall Balanced Accuracy: 0.976365874975245


# Save CSV

In [34]:
# SAVE OOF TO DISK FOR ENSEMBLES
df_oof = pd.DataFrame({'tabtransformer': final_oof_probs.flatten()})
df_oof.to_csv(f'{NAME}_oof.csv', index=False)

df_test = pd.DataFrame({'tabtransformer': final_test_probs.flatten()})
df_test.to_csv(f'{NAME}_test.csv', index=False)

print("Saved oof and test predictions to file")

Saved oof and test predictions to file


In [35]:
test.shape

(270000, 206)

In [36]:
sub = pd.DataFrame({
    'id': test_ids,
    # TARGET: np.argmax(pred, axis=1)
    TARGET: final_test_preds
})
sub[TARGET] = sub[TARGET].map({0: 'Low', 1: 'Medium', 2: 'High'})

sub.to_csv(f'{NAME}_submission.csv', index=False)
sub

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Medium
269996,899996,Low
269997,899997,Medium
269998,899998,Low
